In [1]:
!pwd

/home/brian/Desktop/IA/churn_pred/proyect/notebooks


In [2]:
#!/bin/bash
!curl -L -o ./../data/telco-customer-churn.zip \
  https://www.kaggle.com/api/v1/datasets/download/blastchar/telco-customer-churn

!unzip ./../data/telco-customer-churn.zip -d ./../data/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  171k  100  171k    0     0  92105      0  0:00:01  0:00:01 --:--:--  312k
Archive:  ./../data/telco-customer-churn.zip
replace ./../data/WA_Fn-UseC_-Telco-Customer-Churn.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import joblib
import os
from datetime import datetime


from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC


In [ ]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"Yes": 1, "No": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=777, stratify=y
)

X_eval, X_test, y_eval, y_test = train_test_split(
    X_test, y_test, test_size=0.5, random_state=777, stratify=y_test
)

In [ ]:
print(X_train.shape)
print(X_eval.shape)
print(X_test.shape)


(4930, 19)
(1056, 19)
(1057, 19)


In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_eval_processed = preprocessor.transform(X_eval)


## jugar con distintoas modelos!!! y elegir el mejor

In [ ]:
def optimize_model(model, param_dist, X_train, y_train, X_test, y_test, model_name):
    """Función genérica para optimizar y evaluar cualquier modelo"""
    
    print(f"\n=== Optimizando {model_name} ===")
    
    # Optimización de hiperparámetros
    random_search = RandomizedSearchCV(
        estimator           = model,
        param_distributions = param_dist,
        n_iter              = 50,
        cv                  = 5,
        scoring             = 'roc_auc',
        random_state        = 777,
        n_jobs              = -1
    )
    
    random_search.fit(X_train, y_train)
    
    # Mejor modelo
    best_model = random_search.best_estimator_
    
    # Predicciones
    y_pred = best_model.predict(X_test)
    
    # Resultados
    print(f"Mejores parámetros: {random_search.best_params_}")
    print(f"Mejor score CV: {random_search.best_score_:.4f}")
    print(f"Accuracy test: {best_model.score(X_test, y_test):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    return best_model, random_search.best_score_

In [ ]:

# Definir modelos y sus espacios de parámetros
models_config = [
    {
        'name': 'Random Forest',
        'model': RandomForestClassifier(random_state=777, class_weight='balanced'),
        'params': {
            'n_estimators': np.arange(100, 500, 50),
            'max_depth': [10, 20, 30, None],
            'min_samples_split': np.arange(2, 20),
            'min_samples_leaf': np.arange(1, 10)
        }
    },
    {
        'name': 'Gradient Boosting',
        'model': GradientBoostingClassifier(random_state=777),
        'params': {
            'n_estimators': np.arange(100, 500, 50),
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5, 7],
            'subsample': [0.8, 0.9, 1.0]
        }
    },
    {
        'name': 'Logistic Regression',
        'model': LogisticRegression(random_state=777, max_iter=1000, class_weight='balanced'),
        'params': {
            'C': np.logspace(-1, 2, 20),
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga']
        }
    }
]

In [ ]:
# Lista para guardar resultados
results = []

# Optimizar cada modelo
for config in models_config:
    best_model, cv_score = optimize_model(
        model=config['model'],
        param_dist=config['params'],
        X_train=X_train_processed,
        y_train=y_train,
        X_test=X_eval_processed,
        y_test=y_eval,
        model_name=config['name']
    )
    
    results.append({
        'model': config['name'],
        'best_model': best_model,
        'cv_score': cv_score
    })

# Encontrar el mejor modelo
best_result = max(results, key=lambda x: x['cv_score'])
print(f"\n🏆 Mejor modelo: {best_result['model']} con CV score: {best_result['cv_score']:.4f}")


=== Optimizando Random Forest ===
Mejores parámetros: {'n_estimators': 150, 'min_samples_split': 18, 'min_samples_leaf': 1, 'max_depth': None}
Mejor score CV: 0.8367
Accuracy test: 0.7585

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.77      0.82       776
           1       0.53      0.74      0.62       280

    accuracy                           0.76      1056
   macro avg       0.71      0.75      0.72      1056
weighted avg       0.80      0.76      0.77      1056


=== Optimizando Gradient Boosting ===
Mejores parámetros: {'subsample': 0.8, 'n_estimators': 250, 'max_depth': 3, 'learning_rate': 0.1}
Mejor score CV: 0.8480
Accuracy test: 0.8030

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.91      0.87       776
           1       0.67      0.51      0.58       280

    accuracy                           0.80      1056
   macro avg       0.75      0.71   

/home/brian/Desktop/IA/churn_pred/venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/brian/Desktop/IA/churn_pred/venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/brian/Desktop/IA/churn_pred/venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/brian/Desktop/IA/churn_pred/venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/brian/Desktop/IA/churn_pred/venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  

Mejores parámetros: {'solver': 'liblinear', 'penalty': 'l1', 'C': 1.2742749857031335}
Mejor score CV: 0.8438
Accuracy test: 0.7756

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.79      0.84       776
           1       0.56      0.75      0.64       280

    accuracy                           0.78      1056
   macro avg       0.73      0.77      0.74      1056
weighted avg       0.81      0.78      0.78      1056


🏆 Mejor modelo: Gradient Boosting con CV score: 0.8480


In [6]:

MODEL_PATH = "../models/model_v1.pkl"

def load_data(path):
    df = pd.read_csv(path)
    df = df.dropna()
    return df

def build_pipeline(num_cols, cat_cols):
    best_paramns = {'subsample': 0.9, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(**best_paramns))
    ])

    return model

def main():
    df = load_data("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

    y = df["Churn"].map({"Yes": 1, "No": 0})
    X = df.drop(columns=["Churn", "customerID"], errors="ignore")

    num_cols = X.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = X.select_dtypes(include=["object"]).columns


    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, random_state=777, stratify=y
    )

    model = build_pipeline(num_cols, cat_cols)

    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, model.predict(X_test))
    roc = roc_auc_score(y_test, y_pred)
    f1 = f1_score(y_test, model.predict(X_test))

    print(f"ROC-AUC: {roc:.4f}")
    print(f"F1-Score: {f1:.4f}")

    # Guardar con metadata
    os.makedirs("models", exist_ok=True)

    artifact = {
        "model": model,
        "version": "v1",
        "metrics": {
            "roc_auc": roc,
            "accuracy": accuracy,
            "f1": f1
        },
        "timestamp": datetime.now().isoformat()
    }

    joblib.dump(artifact, MODEL_PATH)

    print(f"Model saved to {MODEL_PATH}")
    print(model.model_version())

if __name__ == "__main__":
    main()

ROC-AUC: 0.8408
F1-Score: 0.5514
Model saved to ../models/model_v1.pkl


AttributeError: 'Pipeline' object has no attribute 'model_version'

In [ ]:
display(X_test.iloc[0])
display(y_test.iloc[0])

gender                        Female
SeniorCitizen                      0
Partner                          Yes
Dependents                       Yes
tenure                            72
PhoneService                      No
MultipleLines       No phone service
InternetService                  DSL
OnlineSecurity                    No
OnlineBackup                     Yes
DeviceProtection                 Yes
TechSupport                      Yes
StreamingTV                      Yes
StreamingMovies                  Yes
Contract                    Two year
PaperlessBilling                 Yes
PaymentMethod       Electronic check
MonthlyCharges                  60.0
TotalCharges                  4264.0
Name: 321, dtype: object

0

prueba de json para predict

In [ ]:
{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "Yes",
    "tenure": 72,
    "PhoneService": "No",
    "MultipleLines": "No phone service",
    "InternetService": "DSL",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "Yes",
    "TechSupport": "Yes",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Two year",
    "PaperlessBilling": "Yes",    
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 60.0,
    "TotalCharges": 4264.0
}

In [ ]:
!curl -X 'POST' \
  'http://127.0.0.1:8000/predict' \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{"gender": "Female","SeniorCitizen": 0,"Partner": "Yes","Dependents": "Yes","tenure": 72,"PhoneService": "No","MultipleLines": "No phone service","InternetService": "DSL","OnlineSecurity": "No","OnlineBackup": "Yes","DeviceProtection": "Yes","TechSupport": "Yes","StreamingTV": "Yes","StreamingMovies": "Yes","Contract": "Two year","PaperlessBilling": "Yes","PaymentMethod": "Electronic check","MonthlyCharges": 60.0,"TotalCharges": 4264.0}'

{"churn_probability":0.05087441216506478,"model_version":"v1"}